# EfficientNet-B0 — MixUp + Rich Histopathology Augmentation

Addresses the ISUP 2/3 confusion (the main bottleneck: 53.5% and 48.1% per-class accuracy
in the best ensemble) using two complementary strategies:

1. **MixUp** (α=0.2) — creates convex combinations of training pairs, forcing the model
   to produce smooth ordinal interpolations between adjacent grades.
2. **Richer augmentation** — adds histopathology-relevant transforms on top of the
   baseline flips: elastic deformation, CLAHE, stain-colour jitter, and blurring.

Loss: same FocalOrdinalLoss as the best baseline.

In [ ]:
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import albumentations as Albu
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import os
import sys
sys.path.append('../../../')
from utils.dataset import PandasDataset
from utils.models import EfficientNetApi
from utils.metrics import calculate_metrics, format_metrics
from utils.train import train_model
from utils.metrics import model_checkpoint

In [ ]:
SEED = 42
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

BATCH_SIZE    = 3
NUM_WORKERS   = 4
OUTPUT_CLASSES = 5
INIT_LR       = 3e-4
WARMUP_FACTOR = 2
WARMUP_EPOCHS = 1
N_EPOCHS      = 50
DROPOUT_RATE  = 0.6
PATIENCE      = 7
MIXUP_ALPHA   = 0.2
MIXUP_PROB    = 0.5   # probability of applying MixUp per batch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

ROOT_DIR   = '../../..'
DATA_DIR   = '../../../..'
IMAGES_DIR = os.path.join(DATA_DIR, 'tiles')
os.makedirs('logs', exist_ok=True)
os.makedirs('models', exist_ok=True)

MODEL_PATH = 'models/b0-mixup-augmented.pth'
LOG_PATH   = 'logs/b0-mixup-augmented.txt'

In [ ]:
class FocalOrdinalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, ordinal_weight=1.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.ordinal_weight = ordinal_weight

    def forward(self, logits, targets):
        targets = targets.to(logits.device).float()
        bce  = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt   = torch.where(targets == 1, probs, 1 - probs)
        focal_loss = (self.alpha * (1 - pt) ** self.gamma * bce).mean()
        ordinal_loss = ((probs.sum(1) - targets.sum(1)) ** 2).mean() / (logits.shape[1] ** 2)
        return focal_loss + self.ordinal_weight * ordinal_loss

LOSS_FN = FocalOrdinalLoss()
print('Loss ready')

In [ ]:
# ── Rich histopathology augmentation ─────────────────────────────────
# Adds elastic + CLAHE + colour jitter + blur on top of baseline flips.
# ElasticTransform simulates tissue deformation (common in pathology).
# HueSaturationValue simulates stain-colour variability between scanners.
TRAIN_TRANSFORMS = Albu.Compose([
    # Geometric
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    Albu.RandomRotate90(p=0.5),
    Albu.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.3),
    Albu.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
    # Intensity / stain
    Albu.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    Albu.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
    Albu.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
    # Noise / blur
    Albu.GaussianBlur(blur_limit=(3, 5), p=0.2),
    Albu.GaussNoise(p=0.2),
])
print('Train transforms defined')

In [ ]:
# ── Data ─────────────────────────────────────────────────────────────
df_all = pd.read_csv(f'{ROOT_DIR}/data/train_5fold.csv')
df_all.columns = df_all.columns.str.strip()

df_entropy = pd.read_csv(f'{ROOT_DIR}/data/entropy.csv')
hard_ids = set(df_entropy.sort_values('difficulty_score', ascending=False)
                         .head(int(len(df_entropy) * 0.2))['image_id'])
df_all = df_all[~df_all['image_id'].isin(hard_ids)].reset_index(drop=True)

df_all  = df_all[df_all['image_id'].apply(
    lambda x: os.path.isfile(f'{IMAGES_DIR}/{x}.png'))].reset_index(drop=True)

df_train = df_all[df_all['fold'] != 3].reset_index(drop=True)
df_val   = df_all[df_all['fold'] == 3].reset_index(drop=True)
df_test  = pd.read_csv(f'{ROOT_DIR}/data/test.csv')
df_test  = df_test[df_test['image_id'].apply(
    lambda x: os.path.isfile(f'{IMAGES_DIR}/{x}.png'))].reset_index(drop=True)

print(f'Train: {len(df_train)}  Val: {len(df_val)}  Test: {len(df_test)}')

In [ ]:
train_ds = PandasDataset(IMAGES_DIR, df_train, transforms=TRAIN_TRANSFORMS, format='png')
valid_ds = PandasDataset(IMAGES_DIR, df_val,   transforms=None,             format='png')
test_ds  = PandasDataset(IMAGES_DIR, df_test,  transforms=None,             format='png')

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                      sampler=RandomSampler(train_ds))
valid_dl = DataLoader(valid_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                      sampler=RandomSampler(valid_ds))
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)

In [ ]:
# ── MixUp training step ───────────────────────────────────────────────
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def training_step_mixup(model, dataloader, optimizer, device, loss_fn, mixup_prob):
    model.train()
    train_losses = []
    bar = tqdm(dataloader, desc='Train')
    for batch_x, batch_y, _ in bar:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()

        if random.random() < mixup_prob:
            mixed_x, y_a, y_b, lam = mixup_data(batch_x, batch_y, MIXUP_ALPHA)
            logits = model(mixed_x)
            loss = lam * loss_fn(logits, y_a) + (1 - lam) * loss_fn(logits, y_b)
        else:
            logits = model(batch_x)
            loss = loss_fn(logits, batch_y)

        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        smooth = sum(train_losses[-100:]) / min(len(train_losses), 100)
        bar.set_postfix(loss=f'{loss.item():.5f}', smooth=f'{smooth:.5f}')
    return train_losses

def validation_step(model, dataloader, device, loss_fn):
    model.eval()
    val_losses, all_preds, all_targets = [], [], []
    with torch.no_grad():
        for batch_x, batch_y, _ in dataloader:
            batch_x = batch_x.to(device); batch_y_d = batch_y.to(device)
            logits = model(batch_x)
            val_losses.append(loss_fn(logits, batch_y_d).item())
            preds = (torch.sigmoid(logits) > 0.5).sum(1).cpu()
            all_preds.append(preds)
            all_targets.append(batch_y.sum(1).long())
    all_preds   = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    from sklearn.metrics import cohen_kappa_score, accuracy_score
    return {
        'val_loss':  np.mean(val_losses),
        'val_kappa': {'mean': cohen_kappa_score(all_targets, all_preds, weights='quadratic')},
        'val_acc':   {'mean': accuracy_score(all_targets, all_preds)},
    }

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────
backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model    = EfficientNetApi(model=backbone, output_dimensions=OUTPUT_CLASSES,
                           dropout_rate=DROPOUT_RATE).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=INIT_LR / WARMUP_FACTOR)
sched_cos = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, N_EPOCHS - WARMUP_EPOCHS)
scheduler = GradualWarmupScheduler(optimizer, multiplier=WARMUP_FACTOR,
                                   total_epoch=WARMUP_EPOCHS, after_scheduler=sched_cos)
print('Model ready')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────
best_kappa = 0.0
no_improve = 0
history = {'train_loss': [], 'val_loss': [], 'val_kappa': []}

for epoch in range(1, N_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{N_EPOCHS}')
    train_losses = training_step_mixup(model, train_dl, optimizer, DEVICE, LOSS_FN, MIXUP_PROB)
    metrics      = validation_step(model, valid_dl, DEVICE, LOSS_FN)
    scheduler.step()

    kappa = metrics['val_kappa']['mean']
    lr    = optimizer.param_groups[0]['lr']
    log   = (f'epoch: {epoch} | lr: {lr:.7f} | train_loss: {np.mean(train_losses):.5f} '
             f'| val_loss: {metrics["val_loss"]:.5f} | val_acc: {metrics["val_acc"]["mean"]:.4f} '
             f'| val_kappa: {kappa:.4f}')
    print(log)
    with open(LOG_PATH, 'a') as f:
        f.write(log + '\n')

    history['train_loss'].append(np.mean(train_losses))
    history['val_loss'].append(metrics['val_loss'])
    history['val_kappa'].append(kappa)

    if kappa >= best_kappa:
        best_kappa = kappa
        no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  ✓ Best saved: {best_kappa:.4f}')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stop. Best kappa: {best_kappa:.4f}')
            break

print(f'Training done. Best val kappa: {best_kappa:.4f}')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history['val_kappa'], color='orange', label='Val Kappa')
axes[1].set_title('Kappa'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig('logs/b0-mixup-augmented-training.png', dpi=300)
plt.show()

In [ ]:
# ── Evaluate on test set ─────────────────────────────────────────────
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for batch_x, batch_y, _ in tqdm(test_dl, desc='Testing'):
        logits = model(batch_x.to(DEVICE))
        all_preds.append((torch.sigmoid(logits) > 0.5).sum(1).cpu())
        all_targets.append(batch_y.sum(1).long())

all_preds   = torch.cat(all_preds).numpy()
all_targets = torch.cat(all_targets).numpy()

metrics = calculate_metrics(all_preds, all_targets)
result  = format_metrics(metrics)
print('\n=== B0-MIXUP-AUGMENTED TEST RESULTS ===')
print(result)

with open('logs/b0-mixup-augmented-test-results.txt', 'w') as f:
    f.write('EfficientNet-B0 — MixUp + Rich Augmentation\n')
    f.write('=' * 70 + '\n\n')
    f.write(result + '\n\n')
    f.write('Classification Report:\n')
    f.write(classification_report(all_targets, all_preds,
                                  target_names=[f'ISUP {i}' for i in range(6)]))
    cm = confusion_matrix(all_targets, all_preds)
    f.write('\nConfusion Matrix:\n' + str(cm))
print('Saved → logs/b0-mixup-augmented-test-results.txt')

In [ ]:
cm      = confusion_matrix(all_targets, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
labels  = [f'ISUP {i}' for i in range(6)]
plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('B0 MixUp + Rich Augmentation — Normalised Confusion Matrix')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('logs/b0-mixup-augmented-confusion-matrix-normalized.png', dpi=300)
plt.show()